# Notebook 4 — Multi-Layer Perceptron on Tabular Data (R4)
Rice Variety Classification | F20DL 2026/2027

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report
from ucimlrepo import fetch_ucirepo
import matplotlib.pyplot as plt

rice = fetch_ucirepo(id=545)
X = rice.data.features.values.astype(np.float32)
y = LabelEncoder().fit_transform(rice.data.targets['Class']).astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=64)

## 1. Define MLP

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(0.3)]
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

model = MLP(input_dim=7, hidden_dims=[64, 32], output_dim=2)
print(model)

## 2. Train

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

train_losses, val_accs = [], []
for epoch in range(50):
    model.train()
    epoch_loss = 0
    for xb, yb in train_dl:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_dl))

    model.eval()
    correct = 0
    with torch.no_grad():
        for xb, yb in test_dl:
            preds = model(xb).argmax(1)
            correct += (preds == yb).sum().item()
    val_accs.append(correct / len(test_ds))
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}: loss={train_losses[-1]:.4f}, acc={val_accs[-1]:.4f}')

## 3. Evaluate & Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses); ax1.set_title('Training Loss')
ax2.plot(val_accs); ax2.set_title('Validation Accuracy')
plt.tight_layout(); plt.savefig('../results/mlp_training.png', dpi=100); plt.show()

model.eval()
all_preds = []
with torch.no_grad():
    for xb, _ in test_dl:
        all_preds.extend(model(xb).argmax(1).numpy())
print(classification_report(y_test, all_preds, target_names=['Cammeo', 'Osmancik']))
torch.save(model.state_dict(), '../models/mlp.pt')